In [2]:
import pathlib
from collections import OrderedDict
from typing import Any, Iterable

import ray
import numpy as np
import anndata
import pandas as pd
# from bolero.tl.dataset.sc_transforms import FilterRegions
# from bolero.tl.dataset.transforms import (
#     FetchRegionOneHot,
#     ReverseComplement,
# )

# from bolero.utils import get_global_coords, understand_regions
# from bolero.tl.dataset.file_transforms import FetchRegionBigWigs, GetEmbedding
# from utils import BorzoiRegions, clamp_sqrt_large_value

# DNA_NAME = "dna_one_hot"
from bolero.tl.model.borzoi.dataset import BorzoiDatasetOnline
from bolero import init

init(num_cpus=32, object_store_memory_gb=100)

2024-10-24 14:08:31,655	INFO worker.py:1762 -- Started a local Ray instance. View the dashboard at 127.0.0.1:8265 


In [3]:
data_dir = '/large_storage/zhoulab/tlgallent/data/Li2023Science/'
bw_file_path = f'{data_dir}bigwig/'
cell_types = ['ASC', 'Vip', 'L5_IT']
# cell_types = ['ASC']
# bed = f'{data_dir}peaks/cCREs.bed'


In [4]:
bigwig_paths = [f'{bw_file_path}'+f'{ct}.bw' for ct in cell_types]
bigwig_paths

['/large_storage/zhoulab/tlgallent/data/Li2023Science/bigwig/ASC.bw',
 '/large_storage/zhoulab/tlgallent/data/Li2023Science/bigwig/Vip.bw',
 '/large_storage/zhoulab/tlgallent/data/Li2023Science/bigwig/L5_IT.bw']

In [5]:
bigwig_names_dict = OrderedDict()
for path in bigwig_paths:
    bigwig_names_dict[pathlib.Path(path).name.rsplit('.',1)[0]] = path


assert len(bigwig_names_dict) == len(bigwig_paths)

In [6]:
for i, x in enumerate(bigwig_names_dict):
    print(i, x)
    print(bigwig_names_dict[x])

0 ASC
/large_storage/zhoulab/tlgallent/data/Li2023Science/bigwig/ASC.bw
1 Vip
/large_storage/zhoulab/tlgallent/data/Li2023Science/bigwig/Vip.bw
2 L5_IT
/large_storage/zhoulab/tlgallent/data/Li2023Science/bigwig/L5_IT.bw


In [7]:
bigwig_paths_dict = OrderedDict()

In [8]:
for path in bigwig_paths:
    cell_type = path.split('/')[-1]
    cell_type = cell_type.rsplit('.',1)[0]
    bigwig_paths_dict[cell_type] = path

In [9]:
path = "/large_storage/zhoulab/tlgallent/data/Li2023Science/bigwig/ASC.bw"

# Remove the leading and trailing quotes
# path = path.strip("'")

# Split the path and get the last part
filename_with_extension = path.split('/')[-1]

# Remove the extension
filename_without_extension = filename_with_extension.rsplit('.', 1)[0]

filename_without_extension

'ASC'

In [10]:
adata = anndata.read_h5ad('/large_storage/zhoulab/tlgallent/data/cell_29000_rna_raw_gencode_adata_with_embeddings.h5ad')
scvi_embedding = adata.obsm['X_scVI']
# Create a DataFrame with embeddings and cell types
df = pd.DataFrame(scvi_embedding, index=adata.obs.index)
df['cell_type'] = adata.obs['MajorType']
# Group by cell type
grouped = df.groupby('cell_type').mean()
# Get embedding
recalculated_embedding = grouped.to_numpy()


leg_map = {item: index for index, item in enumerate(grouped.index.to_list())}
leg_map

/tmp/ipykernel_437424/135511446.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = df.groupby('cell_type').mean()


{'ASC': 0,
 'Amy': 1,
 'CHD7': 2,
 'EC': 3,
 'Foxp2': 4,
 'L4_IT': 5,
 'L5_ET': 6,
 'L5_IT': 7,
 'L6_CT': 8,
 'L6_IT': 9,
 'L6_IT_Car3': 10,
 'L6b': 11,
 'L23_IT': 12,
 'L56_NP': 13,
 'Lamp5': 14,
 'Lamp5_LHX6': 15,
 'MGC': 16,
 'MSN_D1': 17,
 'MSN_D2': 18,
 'ODC': 19,
 'OPC': 20,
 'PC': 21,
 'Pvalb': 22,
 'Pvalb_ChC': 23,
 'Sncg': 24,
 'Sst': 25,
 'SubCtx': 26,
 'VLMC': 27,
 'Vip': 28}

In [11]:
for x in cell_types: print(recalculated_embedding[leg_map[x]])

[ 8.1639951e-01 -1.8141804e-02  1.9753073e-01 -7.8901976e-02
 -4.7131974e-02  3.6194551e-01  4.6953613e-01 -5.1410848e-01
  3.9528120e-01  3.6116695e-01  3.5134324e-01 -3.3298790e-01
  8.2313013e-01 -1.3897213e-04 -2.4195077e-01 -1.2614359e+00
 -1.4791901e-02  8.7709546e-01 -5.9044462e-01  5.2708077e-01
  1.1471124e+00  7.0392508e-03  8.5427901e-03  1.2751074e+00
 -9.8622002e-02 -1.2545434e-02 -1.3628805e-01  3.1145185e-01
  8.4211981e-01 -3.4372029e-01  2.4858487e-01 -2.3774694e-01
 -1.6763769e+00 -3.4681487e-01 -4.7062245e-01  6.3714623e-01
  2.5193891e-01  1.8790249e+00  1.5217294e+00  3.6817715e-01
 -1.1824822e+00  3.0063364e-01 -3.6512426e-01  4.6925202e-01
 -6.6868114e-01  2.3944591e-01  9.7631979e-01 -4.0132409e-01
  4.8463538e-01  6.4016777e-01]
[-6.93263650e-01  2.43766513e-02  5.12231112e-01 -1.10374182e-01
  3.21036965e-01  4.99607325e-02 -5.97075462e-01  5.64686716e-01
 -1.59261763e-01 -2.85740286e-01  2.75409430e-01  6.66004956e-01
  2.10795522e-01  1.94720123e-02 -6.71250

In [12]:
# borzoi_online_dataset = BorzoiDatasetOnline(bigwig_paths=bigwig_paths, bed=bed, leg_map=leg_map, genome="hg38")

In [13]:
# borzoi_online_dataset.train()

In [14]:
# borzoi_online_dataset.get_default_config()

In [15]:
# vars(borzoi_online_dataset)

In [16]:
# loader1 = borzoi_online_dataset.get_dataloader(folds=0, region_bed=bed)

# for batch in loader1:
#     pass

# for k, v in batch.items():
#     print(k, v.dtype, v.shape)
# del loader1

In [17]:
# leg_map['ASC'] = 1841
leg_map

{'ASC': 0,
 'Amy': 1,
 'CHD7': 2,
 'EC': 3,
 'Foxp2': 4,
 'L4_IT': 5,
 'L5_ET': 6,
 'L5_IT': 7,
 'L6_CT': 8,
 'L6_IT': 9,
 'L6_IT_Car3': 10,
 'L6b': 11,
 'L23_IT': 12,
 'L56_NP': 13,
 'Lamp5': 14,
 'Lamp5_LHX6': 15,
 'MGC': 16,
 'MSN_D1': 17,
 'MSN_D2': 18,
 'ODC': 19,
 'OPC': 20,
 'PC': 21,
 'Pvalb': 22,
 'Pvalb_ChC': 23,
 'Sncg': 24,
 'Sst': 25,
 'SubCtx': 26,
 'VLMC': 27,
 'Vip': 28}

## PROCESSED DATA ##

In [18]:
borzoi_online_dataset = BorzoiDatasetOnline(bigwig_paths=bigwig_paths, embeddings_path='/large_storage/zhoulab/tlgallent/data/cell_29000_rna_raw_gencode_adata_with_embeddings.h5ad', bed=None, genome="hg38", use_borzoi_regions=True)
borzoi_online_dataset.train()

/home/tlgallent/projects/software/bolero/src/bolero/tl/model/borzoi/dataset.py:494: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = df.groupby('cell_type').mean()


In [19]:
borzoi_online_dataset.get_default_config()

{'bigwig_paths': 'REQUIRED',
 'bed': 'REQUIRED',
 'embeddings_path': 'REQUIRED',
 'lora': 'REQUIRED',
 'genome': 'hg38',
 'batch_size': 2,
 'dna_window': 524288,
 'pos_resolution': 32,
 'reverse_complement': True,
 'max_jitter': 0,
 'clamp_sqrt_threshold': None,
 'shuffle_files': True,
 'use_borzoi_regions': True}

In [20]:
borzoi_online_dataset.lora

bool

In [21]:
vars(borzoi_online_dataset)

{'genome': Genome: hg38
 Fasta Path: /home/tlgallent/projects/software/bolero/src/bolero/data/hg38/fasta/hg38.fa
 Genome One Hot Zarr: Not created,
 'batch_size': 2,
 'dna': False,
 '_block_size': 20,
 '_max_blocks': 200,
 'bigwig_paths': ['/large_storage/zhoulab/tlgallent/data/Li2023Science/bigwig/ASC.bw',
  '/large_storage/zhoulab/tlgallent/data/Li2023Science/bigwig/Vip.bw',
  '/large_storage/zhoulab/tlgallent/data/Li2023Science/bigwig/L5_IT.bw'],
 'bigwig_names_dict': OrderedDict([('ASC',
               '/large_storage/zhoulab/tlgallent/data/Li2023Science/bigwig/ASC.bw'),
              ('Vip',
               '/large_storage/zhoulab/tlgallent/data/Li2023Science/bigwig/Vip.bw'),
              ('L5_IT',
               '/large_storage/zhoulab/tlgallent/data/Li2023Science/bigwig/L5_IT.bw')]),
 'bigwig_id_dict': OrderedDict([('ASC', 0), ('Vip', 1), ('L5_IT', 2)]),
 'embeddings_path': '/large_storage/zhoulab/tlgallent/data/cell_29000_rna_raw_gencode_adata_with_embeddings.h5ad',
 'leg_map':

In [22]:
# borzoi_online_dataset.dataset_mode

In [23]:
(train_folds, valid_folds, test_folds, train_regions, valid_regions, test_regions) = borzoi_online_dataset.get_train_valid_test(fold=0)

In [24]:
train_regions

,Chromosome,Start,End,Name,Original_Name,fold
0,chr1,155957121,156481409,chr1:155957121-156481409,13844,2
1,chr1,152760876,153285164,chr1:152760876-153285164,13856,2
2,chr1,152121627,152645915,chr1:152121627-152645915,13912,2
3,chr1,150548091,151072379,chr1:150548091-151072379,13942,2
4,chr1,150351399,150875687,chr1:150351399-150875687,13956,2
...,...,...,...,...,...,...
39798,chrX,4650488,5174776,chrX:4650488-5174776,54865,7
39799,chrX,36285423,36809711,chrX:36285423-36809711,54879,7
39800,chrX,4847180,5371468,chrX:4847180-5371468,55180,7
39801,chrX,4355450,4879738,chrX:4355450-4879738,55197,7


In [25]:
loader1 = borzoi_online_dataset.get_dataloader(folds=0, region_bed=train_regions)

In [26]:
borzoi_online_dataset.bed

,Chromosome,Start,End,region,Original_Name,fold
0,chr1,155957121,156481409,chr1:155957121-156481409,13844,2
1,chr1,152760876,153285164,chr1:152760876-153285164,13856,2
2,chr1,152121627,152645915,chr1:152121627-152645915,13912,2
3,chr1,150548091,151072379,chr1:150548091-151072379,13942,2
4,chr1,150351399,150875687,chr1:150351399-150875687,13956,2
...,...,...,...,...,...,...
39798,chrX,4650488,5174776,chrX:4650488-5174776,54865,7
39799,chrX,36285423,36809711,chrX:36285423-36809711,54879,7
39800,chrX,4847180,5371468,chrX:4847180-5371468,55180,7
39801,chrX,4355450,4879738,chrX:4355450-4879738,55197,7


In [27]:
loader2 = borzoi_online_dataset.get_dataloader(folds=0, region_bed=test_regions)

In [28]:
borzoi_online_dataset.bed

,Chromosome,Start,End,region,Original_Name,fold
0,chr1,236570394,237094682,chr1:236570394-237094682,7148,1
1,chr1,238242276,238766564,chr1:238242276-238766564,7196,1
2,chr1,243307095,243831383,chr1:243307095-243831383,7247,1
3,chr1,245274015,245798303,chr1:245274015-245798303,7268,1
4,chr1,238340622,238864910,chr1:238340622-238864910,7332,1
...,...,...,...,...,...,...
6329,chrX,118890376,119414664,chrX:118890376-119414664,13775,1
6330,chrX,118644511,119168799,chrX:118644511-119168799,13776,1
6331,chrX,122142385,122666673,chrX:122142385-122666673,13784,1
6332,chrX,128633221,129157509,chrX:128633221-129157509,13788,1


In [29]:
borzoi_online_dataset.bed

,Chromosome,Start,End,region,Original_Name,fold
0,chr1,236570394,237094682,chr1:236570394-237094682,7148,1
1,chr1,238242276,238766564,chr1:238242276-238766564,7196,1
2,chr1,243307095,243831383,chr1:243307095-243831383,7247,1
3,chr1,245274015,245798303,chr1:245274015-245798303,7268,1
4,chr1,238340622,238864910,chr1:238340622-238864910,7332,1
...,...,...,...,...,...,...
6329,chrX,118890376,119414664,chrX:118890376-119414664,13775,1
6330,chrX,118644511,119168799,chrX:118644511-119168799,13776,1
6331,chrX,122142385,122666673,chrX:122142385-122666673,13784,1
6332,chrX,128633221,129157509,chrX:128633221-129157509,13788,1


In [30]:
borzoi_online_dataset.bed[ borzoi_online_dataset.bed["Original_Name" ] == 7148]

,Chromosome,Start,End,region,Original_Name,fold
0,chr1,236570394,237094682,chr1:236570394-237094682,7148,1


In [31]:
# counter=  0
# for batch in loader1:
#     counter += 1
#     pass

# print(f'counter: {counter}')
# print('printing batch items...')
# for k, v in batch.items():
#     print(k, v.dtype, v.shape)
# # del loader1

In [32]:
# batch.items()

In [33]:
for idx, batch in enumerate(loader2):
    X = batch["dna_one_hot"]
    embedding = batch.get("cell_type_embedding", None)
    y_true = batch["bw_values"]
    break

print(X, X.shape)
print(embedding, embedding.shape)
print(y_true, y_true.shape)

/home/tlgallent/miniconda/envs/bolero_dev/lib/python3.10/site-packages/numpy/core/fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
2024-10-24 14:09:06,329	INFO streaming_executor.py:108 -- Starting execution of Dataset. Full logs are in /tmp/ray/session_2024-10-24_14-08-28_775293_437424/logs/ray-data
2024-10-24 14:09:06,329	INFO streaming_executor.py:109 -- Execution plan of Dataset: InputDataBuffer[Input] -> AllToAllOperator[Repartition]


Loading genome DNA one-hot encoding...
Created remote one-hot object at ObjectRef(00ffffffffffffffffffffffffffffffffffffff0100000002e1f505)
Data loader kwargs {'batch_size': 2, 'prefetch_batches': 3, 'drop_last': True}


2024-10-24 14:09:39,427	INFO streaming_executor.py:108 -- Starting execution of Dataset. Full logs are in /tmp/ray/session_2024-10-24_14-08-28_775293_437424/logs/ray-data
2024-10-24 14:09:39,427	INFO streaming_executor.py:109 -- Execution plan of Dataset: InputDataBuffer[Input] -> ActorPoolMapOperator[MapBatches(select_columns)->MapBatches(FetchRegionBigWigsReduced)] -> ActorPoolMapOperator[MapBatches(FetchRegionBigWigsReduced)] -> ActorPoolMapOperator[MapBatches(FetchRegionBigWigsReduced)] -> ActorPoolMapOperator[MapBatches(_concat_bw_chunks)->MapBatches(FetchRegionOneHot)] -> TaskPoolMapOperator[MapBatches(ReverseComplement)->MapBatches(_region_to_global_coords)] -> TaskPoolMapOperator[FlatMap(_convert_data)]


tensor([[[False, False, False,  ..., False, False,  True],
         [False, False,  True,  ...,  True,  True, False],
         [False, False, False,  ..., False, False, False],
         [ True,  True, False,  ..., False, False, False]],

        [[False, False, False,  ..., False, False,  True],
         [False, False,  True,  ...,  True,  True, False],
         [False, False, False,  ..., False, False, False],
         [ True,  True, False,  ..., False, False, False]]]) torch.Size([2, 4, 524288])
tensor([[-6.9326e-01,  2.4377e-02,  5.1223e-01, -1.1037e-01,  3.2104e-01,
          4.9961e-02, -5.9708e-01,  5.6469e-01, -1.5926e-01, -2.8574e-01,
          2.7541e-01,  6.6600e-01,  2.1080e-01,  1.9472e-02, -6.7125e-02,
         -1.6400e-01,  9.9672e-03, -1.0936e+00, -5.4624e-01, -1.3277e+00,
          7.5367e-02,  1.8270e-01,  2.4486e-03, -5.2664e-01, -1.2295e+00,
          2.4672e-04, -1.0414e+00, -2.0426e-01,  2.9266e-01, -8.8553e-01,
          1.1628e-01,  1.7349e-01, -4.5958e-02, -1.24

In [ ]:
for k, v in b.items():
    print(k, v.dtype, v.shape)
# del loader1

In [32]:
from bolero import Genome
genome = Genome('hg38')
genome.parse_global_coords(b['region'].cpu().numpy())

,Chromosome,Start,End,Name
0,chr1,149859669,150383957,chr1:149859669-150383957
1,chr1,154973661,155497949,chr1:154973661-155497949


In [42]:
region_index =1 #needs to be 0 or 1
cell_type = next((k for k, v in leg_map.items() if v == b['cell_type_id'][region_index]), None)
print(f'cell type: {cell_type}')
atac_bw_path = borzoi_online_dataset.bigwig_names_dict[cell_type]
print(atac_bw_path)
chrom = "chr1"
start = "154973661"
end = "155497949"

cell type: ASC
/large_storage/zhoulab/tlgallent/data/Li2023Science/bigwig/ASC.bw


In [43]:
import pyBigWig
print(atac_bw_path)
with pyBigWig.open(atac_bw_path) as bw:
    data = np.nan_to_num(bw.values(chrom, int(start), int(end), numpy=True))

data[:129]

/large_storage/zhoulab/tlgallent/data/Li2023Science/bigwig/ASC.bw


array([ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  2.,  0.,  0.,  1.,  0.,
        3.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
        0.,  0.,  0.,  0.,  0.,  1.,  0.,  2.,  0.,  0.,  0.,  0.,  0.,
        0.,  2.,  0.,  0.,  0.,  0.,  0.,  0.,  5.,  0.,  0.,  0.,  0.,
        0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
        0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  1.,  0.,  0.,  0.,
        0.,  0.,  0.,  0.,  0.,  0.,  0.,  2.,  0.,  0.,  0.,  0.,  0.,
        4.,  0.,  3.,  0.,  0.,  0.,  2.,  0.,  0.,  3.,  3.,  1.,  1.,
        0.,  2.,  0.,  3.,  2.,  1.,  8.,  9.,  1.,  0.,  0.,  0.,  0.,
        0.,  1.,  0.,  1.,  0.,  0.,  1.,  0.,  0., 14.,  0.,  3.],
      dtype=float32)

In [44]:
b['bw_values'][region_index,:4]

tensor([0.2188, 0.2812, 0.3125, 1.6562])

In [45]:
for i in range(4):
    print(sum(data[i*32:i*32 + 32])/32)

0.21875
0.28125
0.3125
1.65625


In [37]:
b['dna_one_hot'][region_index,:,:16]

tensor([[ True, False, False,  True,  True,  True,  True, False, False,  True,
         False,  True,  True, False, False, False],
        [False, False, False, False, False, False, False, False,  True, False,
         False, False, False, False,  True,  True],
        [False,  True, False, False, False, False, False, False, False, False,
         False, False, False,  True, False, False],
        [False, False,  True, False, False, False, False,  True, False, False,
          True, False, False, False, False, False]])

In [38]:
s = int(start)
e_small = s+16
e = int(end)
e_small

149859685

In [39]:
import torch
nucleotides = np.array(['A', 'C', 'G', 'T'])

for i in range(16):
    indices = torch.argmax(b['dna_one_hot'][1,:,i].int())
    nucleotide = nucleotides[indices.item()]
    print(f"nucleotide: {nucleotide}\n")

nucleotide: G

nucleotide: A

nucleotide: G

nucleotide: A

nucleotide: A

nucleotide: A

nucleotide: G

nucleotide: G

nucleotide: A

nucleotide: C

nucleotide: A

nucleotide: G

nucleotide: G

nucleotide: A

nucleotide: A

nucleotide: A



In [40]:
# data_key = ('bw_values')
# keys = ['Original_Name', 'bw_values', 'dna_one_hot', 'region']

# for k in keys:
#     print(k)
#     if k not in data_key:
#         print('true')
#     else:
#         print('false')  

In [41]:
batch['cell_type_id']

NameError: name 'batch' is not defined

In [ ]:
batch['bw_values'][0]

## CHECK REGIONS AND DNA ONE HOT MANUAL ##

In [ ]:
list(reversed(sorted(list(batch['bw_values'][1,0,:]))))[:15]

In [ ]:
batch['bw_values'][0,:,:6]

In [ ]:
batch['dna_one_hot'][1,:,:322]

In [ ]:
batch['Original_Name'][0]

In [ ]:
batch['Original_Name'][1]

In [ ]:
batch['region'][1]

In [ ]:
batch['region'][0]

In [ ]:
from bolero import Genome

In [ ]:
genome = Genome('hg38')

In [ ]:
genome.parse_global_coords(batch['region'].cpu().numpy())

In [ ]:
#region 0
train_regions[train_regions['Original_Name'] == 52596]

In [ ]:
#region 1
train_regions[train_regions['Original_Name'] == 52883]

In [ ]:
borzoi_online_dataset.bigwig_names_dict

In [ ]:
atac_bw_path = borzoi_online_dataset.bigwig_names_dict['ASC']
chrom = "chrX"
start = "36432942"
end = "36957230"

In [ ]:
import pyBigWig
with pyBigWig.open(atac_bw_path) as bw:
    data = np.nan_to_num(bw.values(chrom, int(start), int(end), numpy=True))

data[:129]

In [ ]:
batch['bw_values'][0,0,:4]

In [ ]:
for i in range(4):
    print(sum(data[i*32:i*32 + 32])/32)

In [ ]:
atac_bw_path = borzoi_online_dataset.bigwig_names_dict['Vip']
# chrom = "chrX"
# start = "36432942"
# end = "36957230"

In [ ]:
import pyBigWig
with pyBigWig.open(atac_bw_path) as bw:
    data = np.nan_to_num(bw.values(chrom, int(start), int(end), numpy=True))

data[:129]

In [ ]:
batch['bw_values'][0,1,:4]

In [ ]:
for i in range(4):
    print(sum(data[i*32:i*32 + 32])/32)

## CHECK REVERSE COMPLEMENT MANUAL ##

In [ ]:
for k, v in batch.items():
    print(k, v.dtype, v.shape)
    

In [ ]:
batch['dna_one_hot'][0,:,:16]

In [ ]:
s = int(start)
e_small = s+16
e = int(end)
e_small

In [ ]:
import torch
nucleotides = np.array(['A', 'C', 'G', 'T'])

for i in range(16):
    indices = torch.argmax(batch['dna_one_hot'][0,:,i].int())
    nucleotide = nucleotides[indices.item()]
    print(f"nucleotide: {nucleotide}\n")
    

In [ ]:
batch['dna_one_hot'][0,:,1]

In [ ]:
batch['dna_one_hot'][0,:,2]

In [ ]:
batch['dna_one_hot'][0,:,3]

In [ ]:
batch['dna_one_hot'][0,:,4]

In [ ]:
batch['dna_one_hot'][0,:,4]

## TESTING FINETUNE SCRIPT

In [ ]:
import joblib
import numpy as np
import pandas as pd
import torch
from bolero import init
from bolero.tl.model.borzoi.train import BorzoiLoRATrainer
import anndata
import time
from datetime import datetime
import os



def main(config):

    # init(num_cpus=16, object_store_memory_gb=100, verbose=True)
    init(num_cpus=32, verbose=True)   
    print(torch.version.cuda)
    print(torch.__version__)
    print(f'Is CUDA available? {torch.cuda.is_available()}')
    

    
    config = BorzoiLoRATrainer.make_config(**config)
    trainer = BorzoiLoRATrainer(config)
    # breakpoint()
    trainer.train(skip_output_adjust=True)

if __name__ == '__main__':

    
    # use_kv_bottleneck = 1
    use_kv_bottleneck = 0
    use_ema = 0
    context_out = 1
    n_pseudobulks = 2
    fold_id = 0
    lora_preset = "all_conditional"

    # kv_bottleneck = "local"
    kv_bottleneck = None
    n_cycles = 1
    cov_soft_clamp = None
    loss_total_weight = 0.2

    warmup_steps = 10000
    lr = 5e-5
    train_batches = 5000
    val_batches = 1000

    # for quick test
    # warmup_steps = 100
    # lr = 1e-3
    # train_batches = 2000
    # val_batches = 500
    
    name = "testing_loader"
    data_dir = '/large_storage/zhoulab/tlgallent/data/Li2023Science/'
    fold_id = 0
    # num_cell_types_used = 30

    bigwig_dir = os.path.join(data_dir, "bigwig")
    # all_entries = os.listdir(bigwig_dir)
    # bw_files = [entry for entry in all_entries if entry.lower().endswith(".bw")]
    # if len(bw_files) < num_cell_types_used:
    #     print(
    #         f"Warning: Only {len(bw_files)} BigWig (.bw) files found, "
    #         f"but {num_cell_types_used} are required."
    #     )
    # selected_bw_files = bw_files[:num_cell_types_used]

    
    
    cell_types = [
    'ASC', 'Amy', 'CHD7', 'EC', 'Foxp2', 'L4_IT', 'L5_ET',
    'L5_IT', 'L6_CT', 'L6_IT', 'L6_IT_Car3', 'L6b', 'L23_IT',
    'L56_NP', 'Lamp5', 'Lamp5_LHX6', 'MGC', 'MSN_D1', 'MSN_D2',
    'ODC', 'OPC', 'PC', 'Pvalb', 'Pvalb_ChC', 'Sncg', 'Sst',
    'SubCtx', 'VLMC', 'Vip'
    ]
    expected_total_length = len(cell_types)

    leave_out = True

    if leave_out:
        
        cell_types = [
            'ASC', 'Amy', 'CHD7', 'EC', 'Foxp2', 'L4_IT', 'L5_ET',
            'L5_IT', 'L6_IT_Car3', 'L6b', 'L23_IT',
            'L56_NP', 'Lamp5', 'Lamp5_LHX6', 'MSN_D1', 'MSN_D2',
            'ODC', 'PC', 'Pvalb', 'Pvalb_ChC', 'Sncg',
            'SubCtx', 'VLMC', 'Vip'
            ]
        
        left_out = ['L6_IT', 'L6_CT', 'Sst', 'OPC', 'MGC']


        overlap = set(cell_types) & set(left_out)
        
        total_length = len(cell_types) + len(left_out)
        assert total_length == expected_total_length, (
        f"Total length is {total_length}, expected {expected_total_length}."
    )
        assert not overlap, f"Overlap found between lists: {overlap}"


    bw_paths_enumerated = [os.path.join(bigwig_dir, file) + '.bw' for file in cell_types]

    assert len(cell_types) == len(bw_paths_enumerated), "num cell types must be same length as paths enuemrated"
    print(f'Using {len(cell_types)} cell types: {bw_paths_enumerated}')

    checkpoint_path = f"/large_storage/zhoulab/hanliu/240729-WMBRNAModel/05.borzoi/borzoi_pretrain/torch_checkpoints/borzoi.human.f{fold_id}.pt"
    dataset_path = (
        f"{data_dir}"
    )
    vq_records_path = "/home/hanliu/data/240729-WMBRNAModel/02.scVI/scVI-VQ/ALL_vq/vq_records.target_frags_8000000.joblib"
    emb_input_features = 50
    use_ema = False if use_ema == 0 else True
    context_out = False if context_out == 0 else True

    config = {
        # file path and wandb
        "output_dir": "model",
        "savename": name,
        "wandb_project": "borzoi_finetuning",
        "wandb_job_type": "lora",
        "wandb_name": f"{name}-fold-{fold_id}",
        # train
        "start_early_stop_after_epoch": 15,
        "max_epochs": 30,
        "patience": 10,
        "use_amp": True,
        "use_ema": use_ema,
        "train_batches": train_batches,
        "val_batches": val_batches,
        "batch_size": 2,
        "accumulate_grad": 64,
        "lr": lr,
        "warmup_steps": warmup_steps,
        "weight_decay": 1e-8,
        "global_clipnorm": 0.1,
        # dataset
        # "dataset_path": dataset_path,
        "fold_split_id": 0,
        "shuffle_rows": 300,
        "dna_window": 524288,
        "pos_resolution": 32,
        "reverse_complement": True,
        "max_jitter": 3,
        # "n_pseudobulks": n_pseudobulks,
        # LoRA Model
        "base_checkpoint_path": checkpoint_path,
        "emb_input_features": emb_input_features,
        "out_channels": 1,
        "hidden_dim": 256,
        "hidden_layers": 1,
        "output_layer_groups": 4,
        "lora_dropout": 0.01,
        "loss_total_weight": loss_total_weight,
        "lora_preset": lora_preset,
        "context_output": context_out,
        # "lora_norm": "layer",
        # base model dropout:
        "transformer_attn_dropout": 0.0,
        "transformer_pos_dropout": 0.0,
        "transformer_ff_dropout": 0.0,
        "final_conv_dropout": 0.0,
        "final_output_dropout": 0.01,
        # pseudobulk and embedding
        # "vq_records": vq_records_path,
        # "target_cov": 8e6,
        # "use_vq_emb": not use_kv_bottleneck,
        "kv_bottleneck": kv_bottleneck,
        "num_memories": 256,
        "dim_memory": 20,
        "num_memory_codebooks": 2,
        "additional_embs": 1,
        "n_cycles": n_cycles,
        # "cov_soft_clamp": cov_soft_clamp,
        #Online Loader reqs
        'bigwig_paths': bw_paths_enumerated,
        'bed': None,
        'embeddings_path': '/large_storage/zhoulab/tlgallent/data/cell_29000_rna_raw_gencode_adata_with_embeddings.h5ad',
        'lora': True,
        # 'use_borzoi_regions': True,
    }


    main(config)



2024-10-24 15:14:22,912	INFO worker.py:1604 -- Calling ray.init() again after it has already been called.


Using 24 cell types: ['/large_storage/zhoulab/tlgallent/data/Li2023Science/bigwig/ASC.bw', '/large_storage/zhoulab/tlgallent/data/Li2023Science/bigwig/Amy.bw', '/large_storage/zhoulab/tlgallent/data/Li2023Science/bigwig/CHD7.bw', '/large_storage/zhoulab/tlgallent/data/Li2023Science/bigwig/EC.bw', '/large_storage/zhoulab/tlgallent/data/Li2023Science/bigwig/Foxp2.bw', '/large_storage/zhoulab/tlgallent/data/Li2023Science/bigwig/L4_IT.bw', '/large_storage/zhoulab/tlgallent/data/Li2023Science/bigwig/L5_ET.bw', '/large_storage/zhoulab/tlgallent/data/Li2023Science/bigwig/L5_IT.bw', '/large_storage/zhoulab/tlgallent/data/Li2023Science/bigwig/L6_IT_Car3.bw', '/large_storage/zhoulab/tlgallent/data/Li2023Science/bigwig/L6b.bw', '/large_storage/zhoulab/tlgallent/data/Li2023Science/bigwig/L23_IT.bw', '/large_storage/zhoulab/tlgallent/data/Li2023Science/bigwig/L56_NP.bw', '/large_storage/zhoulab/tlgallent/data/Li2023Science/bigwig/Lamp5.bw', '/large_storage/zhoulab/tlgallent/data/Li2023Science/bigwi

wandb: Using wandb-core as the SDK backend. Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: tlg (tlg-arc-institute). Use `wandb login --relogin` to force relogin


Training output layer only
Setting up model from config
Loading base model weights from: /large_storage/zhoulab/hanliu/240729-WMBRNAModel/05.borzoi/borzoi_pretrain/torch_checkpoints/borzoi.human.f0.pt
Total model parameters 195429893, trainable parameters 24157925
EMA is not used.
Training LoRA model


2024-10-24 15:14:51,441	INFO streaming_executor.py:108 -- Starting execution of Dataset. Full logs are in /tmp/ray/session_2024-10-24_14-08-28_775293_437424/logs/ray-data
2024-10-24 15:14:51,442	INFO streaming_executor.py:109 -- Execution plan of Dataset: InputDataBuffer[Input] -> AllToAllOperator[Repartition]


...

- MapBatches(_concat_bw_chunks)->MapBatches(FetchRegionOneHot) 7:   0%|                                       …

- MapBatches(ReverseComplement)->MapBatches(_region_to_global_coords) 8:   0%|                                …

- FlatMap(_convert_data) 9:   0%|                                                                             …

- limit=10002 10:   0%|                                                                                       …

Running 0:   0%|                                                                                              …